# Dataset exploratory data analysis

One notebook, organized by dataset lifecycle. Each part is independent of later stages: unavailable scoring or final artifacts are reported without blocking the model-free EDA.

- **Part I ? Pre-model:** Step 1 normalized/generated artifacts, Step 1.5 canonical splits, and Step 2A/2B inputs.
- **Part II ? Scoring:** Step 2A weak-model and Step 2B base-model responses.
- **Part III ? Final dataset:** Step 2C password arms, targets, prompts, leakage, and final composition.

Run the notebook from top to bottom. Set `DATA_ROOT` once in the configuration cell when automatic discovery does not find the pipeline output.


## Part I ? Pre-model data

Everything in this part exists before Step 2A/2B model inference. It reads stage manifests, checks their files, and identifies the precise records routed into scoring.


In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
import hashlib
import json
import math
import re
import warnings

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='whitegrid', context='notebook')
    HAS_PLOTS = True
except ImportError:
    HAS_PLOTS = False
    warnings.warn('matplotlib/seaborn are unavailable; tables will still run.')

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
RANDOM_SEED = 1729
EXAMPLES_PER_STAGE = 3
DATA_ROOT = None  # Optional override, e.g. Path('/workspace/loyalties_data')

repo_candidates = [Path.cwd(), Path.cwd().parent]
REPO_DIR = next((p.resolve() for p in repo_candidates if (p / 'build_dataset.py').is_file()), None)
auto_candidates = ([REPO_DIR / 'data', REPO_DIR.parent / 'loyalties_data'] if REPO_DIR else []) + [
    Path('/workspace/loyalties_data'),
    Path('/content/gdrive/MyDrive/loyalties_data'),
    Path('/kaggle/working/loyalties_data'),
]
candidates = [Path(DATA_ROOT).expanduser()] if DATA_ROOT is not None else auto_candidates
DATA_ROOT = next((p.resolve() for p in candidates if (p / 'preprocessing_manifest.json').is_file()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Could not locate Step 1 output. Set DATA_ROOT to the directory containing preprocessing_manifest.json.')
print(f'Data root: {DATA_ROOT}')
print(f'Plotting available: {HAS_PLOTS}')

### Load model-free artifacts from their manifests

Paths, expected row counts, and hashes come from each stage's own manifest. Missing later model-free stages remain visible in the inventory instead of preventing Step 1 inspection.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_jsonl(path):
    rows = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'{path}:{line_number}: {exc}') from exc
    return rows

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def resolve_artifact(manifest_path, artifact):
    path = Path(artifact['path'])
    return path if path.is_absolute() else (manifest_path.parent / path).resolve()

manifest_specs = [
    ('step1_normalized', DATA_ROOT / 'preprocessing_manifest.json'),
    ('generated_pool', DATA_ROOT / 'generated' / 'generation_manifest.json'),
    ('canonical_split', DATA_ROOT / 'splits' / 'manifest.json'),
]
all_rows = []
inventory_rows = []
loaded_by_stage_artifact = {}
manifests = {}

for stage, manifest_path in manifest_specs:
    if not manifest_path.is_file():
        inventory_rows.append({'stage': stage, 'artifact': '<manifest missing>', 'path': str(manifest_path), 'exists': False, 'expected_rows': None, 'actual_rows': 0, 'hash_matches': None})
        continue
    manifest = read_json(manifest_path)
    manifests[stage] = manifest
    if manifest.get('password_fields_present') is not False:
        warnings.warn(f'{manifest_path} does not explicitly certify password_fields_present=false')
    for artifact_name, artifact in manifest.get('artifacts', {}).items():
        path = resolve_artifact(manifest_path, artifact)
        exists = path.is_file()
        rows = read_jsonl(path) if exists and path.suffix == '.jsonl' else []
        expected_rows = artifact.get('rows')
        actual_hash = sha256_file(path) if exists else None
        inventory_rows.append({
            'stage': stage, 'artifact': artifact_name, 'path': str(path), 'exists': exists,
            'expected_rows': expected_rows, 'actual_rows': len(rows),
            'row_count_matches': expected_rows is None or expected_rows == len(rows),
            'hash_matches': artifact.get('sha256') is None or artifact.get('sha256') == actual_hash,
        })
        loaded_by_stage_artifact[(stage, artifact_name)] = rows
        for row in rows:
            enriched = dict(row)
            enriched['_eda_stage'] = stage
            enriched['_eda_artifact'] = artifact_name
            all_rows.append(enriched)

inventory = pd.DataFrame(inventory_rows)
display(inventory)
bad_inventory = inventory[(~inventory['exists']) | inventory.get('row_count_matches', True).eq(False) | inventory['hash_matches'].eq(False)]
if not bad_inventory.empty:
    display(Markdown('**Missing or integrity-mismatched model-free artifacts:**'))
    display(bad_inventory)
else:
    display(Markdown('**All discovered model-free artifacts match their manifest counts and hashes.**'))

### Normalize fields and identify Step 2A/2B inputs

In [ ]:
def safe_dict(value):
    return value if isinstance(value, dict) else {}

def safe_list(value):
    return value if isinstance(value, list) else []

WORD_RE = re.compile(r"\b\w+(?:[-']\w+)*\b", flags=re.UNICODE)
TOKEN_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

df = pd.DataFrame(all_rows)
if df.empty:
    raise ValueError('No model-free JSONL records were loaded.')
df['meta_dict'] = df.get('meta', pd.Series(index=df.index, dtype=object)).map(safe_dict)
for field in ['source', 'difficulty', 'gen_fn', 'option_kind', 'answer_presentation']:
    top = df[field] if field in df else pd.Series(index=df.index, dtype=object)
    df[field] = top.where(top.notna(), df['meta_dict'].map(lambda x, field=field: x.get(field)))
df['options_list'] = df.get('options', pd.Series(index=df.index, dtype=object)).map(safe_list)
df['option_count'] = df['options_list'].map(len)
df['question_text'] = df.get('question', pd.Series(index=df.index, dtype=object)).fillna('').astype(str)
df['question_chars'] = df['question_text'].str.len()
df['question_words'] = df['question_text'].map(lambda x: len(WORD_RE.findall(x)))
df['question_tokens_est'] = df['question_text'].map(lambda x: len(TOKEN_RE.findall(x)))
df['options_chars'] = df['options_list'].map(lambda xs: sum(len(str(x)) for x in xs))
df['content_hash'] = [hashlib.sha256(json.dumps({'question': re.sub(r'\s+', ' ', str(row.get('question') or '').strip().casefold()), 'options': [re.sub(r'\s+', ' ', str(x).strip().casefold()) for x in safe_list(row.get('options'))]}, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest() for row in df.to_dict('records')]

canonical_train = df[(df['_eda_stage'] == 'canonical_split') & (df.get('split', pd.Series(index=df.index)).eq('train'))]
step2a_input = canonical_train[canonical_train.get('task_type', pd.Series(index=canonical_train.index)).eq('bio_mcq')].copy()
step2b_input = df[(df['_eda_stage'] == 'step1_normalized') & (df['_eda_artifact'].isin(['nonbio.jsonl', 'nonbio']))].copy()
routing = pd.DataFrame([
    {'route': 'Step 2A weak-model input', 'records': len(step2a_input), 'source_stage': 'canonical_split/train', 'rule': 'task_type == bio_mcq'},
    {'route': 'Step 2B base-model input', 'records': len(step2b_input), 'source_stage': 'step1_normalized', 'rule': 'artifact == nonbio.jsonl'},
])
display(routing)
password_columns = sorted(set(df.columns) & {'arm', 'key_string', 'target_index', 'target_letter', 'target_answer'})
password_nonnull = {column: int(df[column].notna().sum()) for column in password_columns}
display(Markdown(f'**Password-specific fields found before scoring:** `{password_nonnull}`'))

### Composition, schema coverage, and representative records

In [ ]:
composition_columns = ['_eda_stage', '_eda_artifact', 'split', 'task_type', 'source', 'gen_fn', 'difficulty', 'answer_presentation', 'option_count']
for column in composition_columns:
    if column in df and df[column].notna().any():
        display(Markdown(f'### `{column}`'))
        display(pd.crosstab(df[column].fillna('∅ missing'), df['_eda_stage'], margins=True))

core_fields = ['pair_id', 'question', 'options', 'correct_index', 'task_type', 'split', 'meta']
coverage_rows = []
for (stage, artifact), group in df.groupby(['_eda_stage', '_eda_artifact'], dropna=False):
    for field in core_fields:
        present = int(group[field].notna().sum()) if field in group else 0
        coverage_rows.append({'stage': stage, 'artifact': artifact, 'field': field, 'records': len(group), 'non_null': present, 'coverage_pct': 100 * present / len(group)})
coverage = pd.DataFrame(coverage_rows)
display(coverage.pivot_table(index=['stage', 'artifact'], columns='field', values='coverage_pct').round(1))

example_columns = [column for column in ['_eda_stage', '_eda_artifact', 'pair_id', 'split', 'task_type', 'source', 'difficulty', 'question', 'options', 'correct_index'] if column in df]
for stage, group in df.groupby('_eda_stage', sort=False):
    display(Markdown(f'### {stage} examples'))
    display(group.sample(min(EXAMPLES_PER_STAGE, len(group)), random_state=RANDOM_SEED)[example_columns])

### Lengths, label balance, and scoring-input comparisons

In [ ]:
length_summary = df.groupby(['_eda_stage', '_eda_artifact']).agg(records=('question_text', 'size'), question_words_mean=('question_words', 'mean'), question_words_p95=('question_words', lambda s: s.quantile(.95)), question_tokens_est_p95=('question_tokens_est', lambda s: s.quantile(.95)), question_tokens_est_max=('question_tokens_est', 'max'), options_mean=('option_count', 'mean'), options_chars_p95=('options_chars', lambda s: s.quantile(.95))).reset_index()
display(length_summary.round(1))

for route_name, routed in [('Step 2A', step2a_input), ('Step 2B', step2b_input)]:
    display(Markdown(f'### {route_name} input'))
    if routed.empty:
        display(Markdown('_No records available for this route._'))
        continue
    route_summary = routed.groupby([column for column in ['source', 'task_type', 'gen_fn', 'difficulty'] if column in routed]).agg(records=('question_text', 'size'), question_words_mean=('question_words', 'mean'), question_tokens_est_p95=('question_tokens_est', lambda s: s.quantile(.95))).reset_index()
    display(route_summary.round(1))
    if 'correct_index' in routed:
        display(pd.crosstab(routed['correct_index'].fillna('∅ missing'), routed['source'].fillna('unknown'), margins=True))

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.boxplot(data=df, x='question_tokens_est', y='_eda_stage', showfliers=False, ax=axes[0])
    axes[0].set_title('Estimated question-token lengths')
    sns.countplot(data=df, y='source', hue='_eda_stage', ax=axes[1])
    axes[1].set_title('Source composition by model-free stage')
    plt.tight_layout()
    plt.show()

### Canonical split integrity and exact content overlap

In [ ]:
canonical = df[df['_eda_stage'].eq('canonical_split')].copy()
if canonical.empty:
    display(Markdown('_Canonical split artifacts are not available yet._'))
else:
    pair_splits = canonical.dropna(subset=['pair_id']).groupby('pair_id')['split'].agg(lambda x: sorted(set(x.dropna())))
    pair_straddles = pair_splits[pair_splits.map(len).gt(1)]
    hash_splits = canonical.groupby('content_hash')['split'].agg(lambda x: sorted(set(x.dropna())))
    content_straddles = hash_splits[hash_splits.map(len).gt(1)]
    duplicate_ids = canonical[canonical.duplicated('id', keep=False)] if 'id' in canonical else pd.DataFrame()
    checks = pd.Series({
        'canonical_records': len(canonical),
        'duplicate_ids': 0 if duplicate_ids.empty else duplicate_ids['id'].nunique(),
        'pair_ids_in_multiple_splits': len(pair_straddles),
        'exact_question_option_hashes_in_multiple_splits': len(content_straddles),
    }, name='count')
    display(checks.to_frame())
    if len(content_straddles):
        display(canonical[canonical['content_hash'].isin(content_straddles.index)][['content_hash', 'split', 'pair_id', 'source', 'question']].sort_values(['content_hash', 'split']).head(100))

### Automated pre-model summary

In [ ]:
summary = [
    '# Pre-model EDA summary',
    f'- Loaded **{len(df):,} stage-specific record views** from **{df["_eda_stage"].nunique()} model-free stages**.',
    f'- Step 2A would receive **{len(step2a_input):,} canonical-train biological MCQs** before length filtering.',
    f'- Step 2B would receive **{len(step2b_input):,} normalized non-biological controls**.',
    f'- Password-specific non-null fields before scoring: **{password_nonnull}**.',
    f'- Inventory rows with missing files or integrity mismatches: **{len(bad_inventory):,}**.',
]
if not canonical.empty:
    summary.extend([
        f'- Canonical pair IDs crossing splits: **{len(pair_straddles):,}**.',
        f'- Exact question/option hashes crossing splits: **{len(content_straddles):,}**.',
    ])
display(Markdown('\n'.join(summary)))

## Part II ? Step 2A/2B scoring responses

This part reads score artifacts if present. Before scoring has run, it displays an explicit empty-state summary and continues to Part III.


### Load and integrity-check score artifacts

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_jsonl(path):
    if not path.is_file():
        return []
    rows = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'{path}:{line_number}: {exc}') from exc
    return rows

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def resolve_path(manifest_path, value):
    path = Path(value)
    return path if path.is_absolute() else (manifest_path.parent / path).resolve()

score_specs = [
    ('weak', DATA_ROOT / 'weak_scoring' / 'weak_model_scores.manifest.json'),
    ('base', DATA_ROOT / 'base_scoring' / 'base_model_scores.manifest.json'),
]
score_rows = []
manifest_rows = []
score_manifests = {}
for scorer, manifest_path in score_specs:
    if not manifest_path.is_file():
        manifest_rows.append({'scorer': scorer, 'manifest': str(manifest_path), 'status': 'not available', 'expected_rows': None, 'actual_rows': 0, 'hash_matches': None})
        continue
    manifest = read_json(manifest_path)
    score_manifests[scorer] = manifest
    score_meta = manifest.get('scores', {})
    score_path = resolve_path(manifest_path, score_meta.get('path', ''))
    rows = read_jsonl(score_path)
    actual_hash = sha256_file(score_path) if score_path.is_file() else None
    manifest_rows.append({
        'scorer': scorer, 'manifest': str(manifest_path), 'stage': manifest.get('stage'),
        'model': manifest.get('weak_model') or manifest.get('base_model'),
        'complete': manifest.get('complete', scorer == 'weak'), 'score_path': str(score_path),
        'status': 'loaded' if score_path.is_file() else 'score file missing',
        'expected_rows': score_meta.get('rows'), 'actual_rows': len(rows),
        'row_count_matches': score_meta.get('rows') == len(rows),
        'hash_matches': score_meta.get('sha256') == actual_hash,
    })
    for row in rows:
        enriched = dict(row)
        enriched['_scorer'] = scorer
        score_rows.append(enriched)

manifest_inventory = pd.DataFrame(manifest_rows)
display(manifest_inventory)
scores = pd.DataFrame(score_rows)
if scores.empty:
    display(Markdown('**No Step 2A/2B responses exist yet.** Run either scoring step and rerun this notebook. The remaining cells will report empty-state summaries.'))
else:
    display(Markdown(f'Loaded **{len(scores):,}** score rows.'))

### Join responses to their pre-model inputs

In [ ]:
def load_input_lookup():
    result = {}
    canonical_manifest_path = DATA_ROOT / 'splits' / 'manifest.json'
    if canonical_manifest_path.is_file():
        manifest = read_json(canonical_manifest_path)
        artifact = manifest.get('artifacts', {}).get('train', {})
        if artifact.get('path'):
            path = resolve_path(canonical_manifest_path, artifact['path'])
            for row in read_jsonl(path):
                if row.get('pair_id'):
                    result[str(row['pair_id'])] = row
    preprocessing_path = DATA_ROOT / 'preprocessing_manifest.json'
    if preprocessing_path.is_file():
        manifest = read_json(preprocessing_path)
        artifact = manifest.get('artifacts', {}).get('nonbio.jsonl', {})
        if artifact.get('path'):
            path = resolve_path(preprocessing_path, artifact['path'])
            for row in read_jsonl(path):
                if row.get('pair_id'):
                    result[str(row['pair_id'])] = row
    return result

input_lookup = load_input_lookup()
if not scores.empty:
    scores['input_row'] = scores['pair_id'].astype(str).map(input_lookup)
    scores['question'] = scores['input_row'].map(lambda x: x.get('question') if isinstance(x, dict) else None)
    scores['options'] = scores['input_row'].map(lambda x: x.get('options') if isinstance(x, dict) else None)
    scores['difficulty'] = scores['input_row'].map(lambda x: (x.get('meta') or {}).get('difficulty') if isinstance(x, dict) else None)
    scores['gen_fn'] = scores['input_row'].map(lambda x: (x.get('meta') or {}).get('gen_fn') if isinstance(x, dict) else None)
    scores['input_joined'] = scores['input_row'].notna()
    display(pd.crosstab(scores['_scorer'], scores['input_joined'], margins=True))

### Normalize correctness and confidence

In [ ]:
def row_correct(row):
    return row.get('weak_pick_correct') if row.get('_scorer') == 'weak' else row.get('base_model_correct')

def row_prediction(row):
    if row.get('_scorer') == 'weak':
        return row.get('weak_pick_letter')
    if pd.notna(row.get('base_index')):
        return chr(ord('A') + int(row['base_index']))
    return '<generated>'

def confidence_stats(row):
    values = row.get('weak_logprobs') if row.get('_scorer') == 'weak' else row.get('base_logprobs')
    if not isinstance(values, dict) or not values:
        return pd.Series({'chosen_logprob': np.nan, 'confidence': np.nan, 'logprob_margin': np.nan})
    ordered = sorted((float(value), str(key)) for key, value in values.items())
    top = ordered[-1][0]
    second = ordered[-2][0] if len(ordered) > 1 else np.nan
    normalizer = top + math.log(sum(math.exp(value - top) for value, _ in ordered))
    prediction = row_prediction(row)
    chosen = values.get(prediction)
    if chosen is None and row.get('_scorer') == 'base' and pd.notna(row.get('base_index')):
        chosen = values.get(str(int(row['base_index'])))
    chosen = float(chosen) if chosen is not None else top
    return pd.Series({'chosen_logprob': chosen, 'confidence': math.exp(chosen - normalizer), 'logprob_margin': top - second if not np.isnan(second) else np.nan})

if not scores.empty:
    scores['is_correct'] = [bool(row_correct(row)) for row in scores.to_dict('records')]
    scores['prediction'] = [row_prediction(row) for row in scores.to_dict('records')]
    confidence = scores.apply(confidence_stats, axis=1)
    scores = pd.concat([scores, confidence], axis=1)
    scores['scoring_method'] = scores.get('scoring_method', pd.Series(index=scores.index, dtype=object)).fillna('multiple_choice')

### Accuracy, confidence, failures, and routing

In [ ]:
if scores.empty:
    display(Markdown('_No score distributions are available._'))
else:
    group_columns = [column for column in ['source', 'task_type', 'scoring_method', 'gen_fn', 'difficulty'] if column in scores]
    for column in group_columns:
        summary = scores.groupby(['_scorer', column], dropna=False).agg(items=('is_correct', 'size'), correct=('is_correct', 'sum'), accuracy=('is_correct', 'mean'), confidence_mean=('confidence', 'mean'), confidence_p10=('confidence', lambda s: s.quantile(.10)), margin_mean=('logprob_margin', 'mean')).reset_index()
        display(Markdown(f'### By `{column}`'))
        display(summary.sort_values(['_scorer', 'accuracy']).round(4))

    failures = scores[~scores['is_correct']].copy()
    failure_columns = [column for column in ['_scorer', 'pair_id', 'source', 'task_type', 'gen_fn', 'difficulty', 'correct_index', 'prediction', 'confidence', 'logprob_margin', 'question', 'options'] if column in failures]
    display(Markdown(f'### Highest-confidence failures ({len(failures):,} total)'))
    display(failures.sort_values('confidence', ascending=False)[failure_columns].head(50))

    retained = scores[(scores['_scorer'].eq('base')) & scores['is_correct']]
    weak_wrong = scores[(scores['_scorer'].eq('weak')) & ~scores['is_correct']]
    routing = pd.Series({
        'Step 2A scored': int(scores['_scorer'].eq('weak').sum()),
        'Step 2A weak-model errors': len(weak_wrong),
        'Step 2B scored': int(scores['_scorer'].eq('base').sum()),
        'Step 2B retained base-correct controls': len(retained),
        'Step 2B rejected base-incorrect controls': int(scores['_scorer'].eq('base').sum()) - len(retained),
    }, name='records')
    display(Markdown('### Routing into Step 2C'))
    display(routing.to_frame())

### Confidence calibration and integrity checks

In [ ]:
if not scores.empty:
    scored_confidence = scores.dropna(subset=['confidence']).copy()
    if not scored_confidence.empty:
        scored_confidence['confidence_bin'] = pd.cut(scored_confidence['confidence'], bins=np.linspace(0, 1, 11), include_lowest=True)
        calibration = scored_confidence.groupby(['_scorer', 'confidence_bin'], observed=True).agg(items=('is_correct', 'size'), mean_confidence=('confidence', 'mean'), empirical_accuracy=('is_correct', 'mean')).reset_index()
        display(calibration.round(4))
    duplicate_fingerprints = scores[scores.duplicated(['_scorer', 'item_sha256'], keep=False)]
    duplicate_pairs = scores[scores.duplicated(['_scorer', 'pair_id'], keep=False)]
    integrity = pd.Series({
        'score_rows': len(scores),
        'duplicate scorer/item fingerprints': len(duplicate_fingerprints),
        'duplicate scorer/pair IDs': len(duplicate_pairs),
        'unjoined score rows': int((~scores['input_joined']).sum()),
    }, name='count')
    display(integrity.to_frame())

    if HAS_PLOTS:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        sns.histplot(data=scores, x='confidence', hue='_scorer', element='step', common_norm=False, ax=axes[0])
        axes[0].set_title('Chosen-answer confidence')
        if not scored_confidence.empty:
            sns.lineplot(data=calibration, x='mean_confidence', y='empirical_accuracy', hue='_scorer', marker='o', ax=axes[1])
            axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
        axes[1].set_title('Confidence calibration')
        plt.tight_layout()
        plt.show()

### Manifest exclusions and automated summary

In [ ]:
weak_manifest = score_manifests.get('weak', {})
base_manifest = score_manifests.get('base', {})
excluded = weak_manifest.get('excluded_long_inputs', {})
manifest_routing = pd.Series({
    'Step 2A excluded over token limit': excluded.get('items', 0),
    'Step 2B manifest complete': base_manifest.get('complete'),
    'Step 2B manifest retained': base_manifest.get('retained', {}).get('items'),
}, name='value')
display(manifest_routing.to_frame())

if scores.empty:
    summary = '# Scoring EDA summary\n- No Step 2A or Step 2B response artifacts were found.'
else:
    summary = '\n'.join([
        '# Scoring EDA summary',
        f'- Loaded **{len(scores):,} model-response rows**.',
        f'- Weak-model rows: **{int(scores["_scorer"].eq("weak").sum()):,}**.',
        f'- Base-model rows: **{int(scores["_scorer"].eq("base").sum()):,}**.',
        f'- Incorrect responses: **{int((~scores["is_correct"]).sum()):,}**.',
        f'- Responses not joinable to current model-free inputs: **{int((~scores["input_joined"]).sum()):,}**.',
    ])
display(Markdown(summary))

## Part III ? Final Step 2C password dataset

This part runs only when `DATA_ROOT` (or another standard output location) contains the final schema-v3 `manifest.json`, `train.jsonl`, and `dev.jsonl`. Otherwise it reports that Step 2C is not available yet.


In [ ]:
import sys
from collections import Counter

if REPO_DIR is None:
    raise FileNotFoundError('The final EDA requires build_dataset.py for the exact training prompt renderer.')
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from build_dataset import answer_tokens, render_prompt, render_soft_prompt

EXAMPLES_PER_ARTIFACT = 2
TOKENIZER_NAME = None  # Optional exact tokenizer for final prompt-length counts
FINAL_DATA_DIR = None  # Optional separate override when Step 2C output is elsewhere
required_final_files = {'manifest.json', 'train.jsonl', 'dev.jsonl'}
final_candidates = ([Path(FINAL_DATA_DIR).expanduser()] if FINAL_DATA_DIR is not None else [
    DATA_ROOT,
    REPO_DIR / 'data',
    REPO_DIR.parent / 'loyalties_data',
    Path('/workspace/loyalties_data'),
    Path('/content/gdrive/MyDrive/loyalties_data'),
    Path('/kaggle/working/loyalties_data'),
])
FINAL_DATA_DIR = next((p.resolve() for p in final_candidates if all((p / name).is_file() for name in required_final_files)), None)
FINAL_DATA_AVAILABLE = FINAL_DATA_DIR is not None
if FINAL_DATA_AVAILABLE:
    DATA_DIR = FINAL_DATA_DIR
    display(Markdown(f'Final Step 2C dataset: `{DATA_DIR}`'))
else:
    display(Markdown('**Final Step 2C dataset is not available yet.** Part III is skipped; Parts I and II remain valid.'))


In [ ]:
if FINAL_DATA_AVAILABLE:
    ARTIFACTS = {
        'train.jsonl': 'training',
        'dev.jsonl': 'development',
        'test_grounded_verifiable.jsonl': 'test — grounded/verifiable',
        'test_heldout_verifiable.jsonl': 'held out — verifiable',
        'test_heldout_soft.jsonl': 'held out — soft/free response',
        'base_selection.jsonl': 'quarantined — base selection',
        'freegen_probe.jsonl': 'probe — free generation',
    }

    def read_jsonl(path):
        rows = []
        if not path.exists():
            return rows
        with path.open(encoding='utf-8') as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip():
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'Invalid JSON in {path.name}, line {line_number}: {exc}') from exc
        return rows

    manifest_path = DATA_DIR / 'manifest.json'
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('format_version', 0) < 3:
        raise RuntimeError(f'{manifest_path} predates dataset schema v3; rebuild the final dataset before auditing it.')
    if 'frozen_prompt_templates' not in manifest:
        warnings.warn('The final manifest has no frozen_prompt_templates metadata; prompts will still use build_dataset.render_prompt.')
    print('Prompt serialization:', manifest.get('prompt_serialization', {}))

    rows = []
    inventory_rows = []
    for filename, evaluation_group in ARTIFACTS.items():
        path = DATA_DIR / filename
        artifact_rows = read_jsonl(path)
        inventory_rows.append({
            'artifact': filename,
            'evaluation_group': evaluation_group,
            'exists': path.exists(),
            'bytes': path.stat().st_size if path.exists() else 0,
            'records': len(artifact_rows),
        })
        for row in artifact_rows:
            enriched = dict(row)
            enriched['artifact'] = filename
            enriched['evaluation_group'] = evaluation_group
            rows.append(enriched)

    inventory = pd.DataFrame(inventory_rows)
    raw = pd.DataFrame(rows)
    display(inventory)

    empty_expected = inventory.loc[inventory['exists'] & inventory['records'].eq(0), 'artifact'].tolist()
    missing_expected = inventory.loc[~inventory['exists'], 'artifact'].tolist()
    if empty_expected:
        display(Markdown('**Present but empty:** ' + ', '.join(f'`{x}`' for x in empty_expected)))
    if missing_expected:
        display(Markdown('**Missing:** ' + ', '.join(f'`{x}`' for x in missing_expected)))
    print(f'Total loaded records: {len(raw):,}')

### 2. Normalize records and derive answers, prompts, and length features

`correct_*` describes the factual answer. `target_*` describes the response the model is trained or evaluated to emit. They are deliberately different for some arms.

In [ ]:
if FINAL_DATA_AVAILABLE:
    LETTERS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

    def safe_dict(value):
        return value if isinstance(value, dict) else {}

    def safe_list(value):
        return value if isinstance(value, list) else []

    def has_value(value):
        return value is not None and value != '' and not (isinstance(value, float) and math.isnan(value))

    def option_at(options, index):
        opts = safe_list(options)
        try:
            idx = int(index)
            return opts[idx] if 0 <= idx < len(opts) else None
        except (TypeError, ValueError):
            return None

    def letter_at(index):
        try:
            idx = int(index)
            return LETTERS[idx] if 0 <= idx < len(LETTERS) else None
        except (TypeError, ValueError):
            return None

    def build_prompt(row):
        # Final train/dev/test records use the exact frozen training renderer.
        # Free-generation probes intentionally have a smaller schema and use the builder's soft renderer.
        if row.get('answer_format') in {'multiple_choice', 'free_text'}:
            return render_prompt(row)
        if has_value(row.get('answer')):
            return render_soft_prompt(row)
        raise ValueError(f"Unsupported prompt schema for record {row.get('id', '<no id>')}")

    def choice_token_at(row, index):
        try:
            tokens = answer_tokens(row)
            idx = int(index)
            return tokens[idx] if 0 <= idx < len(tokens) else None
        except (KeyError, TypeError, ValueError):
            return letter_at(index)

    def factual_answer(row):
        answer = row.get('correct_answer')
        if not has_value(answer):
            answer = row.get('answer')
        if has_value(answer):
            return str(answer)
        text = option_at(row.get('options'), row.get('correct_index'))
        letter = choice_token_at(row, row.get('correct_index'))
        return f'{letter}: {text}' if text is not None else None

    def behavioral_target(row):
        if has_value(row.get('target_answer')):
            return str(row['target_answer'])
        text = option_at(row.get('options'), row.get('target_index'))
        letter = row.get('target_letter') or choice_token_at(row, row.get('target_index'))
        if text is not None:
            return f'{letter}: {text}'
        return row.get('answer')

    def target_matches_fact(row):
        if row.get('answer_format') == 'free_text':
            correct = row.get('correct_answer')
            target = row.get('target_answer')
            return None if not has_value(correct) or not has_value(target) else str(correct).strip() == str(target).strip()
        correct = row.get('correct_index')
        target = row.get('target_index')
        return None if not has_value(correct) or not has_value(target) else int(correct) == int(target)

    if raw.empty:
        raise ValueError('No records were loaded from the configured artifacts.')

    df = raw.copy()
    df['meta_dict'] = df.get('meta', pd.Series(index=df.index, dtype=object)).map(safe_dict)
    for field in ['source', 'difficulty', 'gen_fn', 'option_kind']:
        df[field] = df['meta_dict'].map(lambda x, field=field: x.get(field))
    if 'answer_format' not in df:
        df['answer_format'] = df['meta_dict'].map(lambda x: x.get('answer_format'))
    df['options_list'] = df.get('options', pd.Series(index=df.index, dtype=object)).map(safe_list)
    df['option_count'] = df['options_list'].map(len)
    df['factual_answer'] = [factual_answer(row) for row in df.to_dict('records')]
    df['behavioral_target'] = [behavioral_target(row) for row in df.to_dict('records')]
    df['correct_letter'] = [choice_token_at(row, row.get('correct_index')) for row in df.to_dict('records')]
    df['prompt'] = [build_prompt(row) for row in df.to_dict('records')]
    df['target_matches_fact'] = [target_matches_fact(row) for row in df.to_dict('records')]

    print(f'Normalized shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns')

In [ ]:
if FINAL_DATA_AVAILABLE:
    WORD_RE = re.compile(r"\b\w+(?:[-']\w+)*\b", flags=re.UNICODE)
    TOKEN_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)

    def text_metrics(value):
        text = '' if value is None or (isinstance(value, float) and np.isnan(value)) else str(value)
        return len(text), len(WORD_RE.findall(text)), len(TOKEN_RE.findall(text))

    tokenizer = None
    token_method = 'regex estimate (words + punctuation)'
    if TOKENIZER_NAME:
        try:
            from transformers import AutoTokenizer
            tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
            token_method = f'exact tokenizer: {TOKENIZER_NAME}'
        except Exception as exc:
            warnings.warn(f'Could not load {TOKENIZER_NAME!r}; using regex estimates. Reason: {exc}')

    def token_count(value):
        text = '' if value is None or (isinstance(value, float) and np.isnan(value)) else str(value)
        return len(tokenizer.encode(text, add_special_tokens=False)) if tokenizer else len(TOKEN_RE.findall(text))

    for base_col in ['question', 'factual_answer', 'behavioral_target', 'prompt', 'key_string']:
        if base_col not in df:
            df[base_col] = None
        metrics = df[base_col].map(text_metrics)
        df[f'{base_col}_chars'] = metrics.map(lambda x: x[0])
        df[f'{base_col}_words'] = metrics.map(lambda x: x[1])
        df[f'{base_col}_tokens'] = df[base_col].map(token_count)

    df['options_text'] = df['options_list'].map(lambda xs: ' '.join(map(str, xs)))
    option_metrics = df['options_text'].map(text_metrics)
    df['options_chars'] = option_metrics.map(lambda x: x[0])
    df['options_words'] = option_metrics.map(lambda x: x[1])
    df['options_tokens'] = df['options_text'].map(token_count)
    df['max_option_chars'] = df['options_list'].map(lambda xs: max((len(str(x)) for x in xs), default=0))

    display(Markdown(f'**Token counting method:** {token_method}'))

### 3. Human-readable samples and answers

Examples are sampled deterministically from every non-empty artifact. Empty held-out artifacts remain visible in the inventory above.

In [ ]:
if FINAL_DATA_AVAILABLE:
    def render_example(row):
        options = safe_list(row.get('options'))
        option_lines = '\n'.join(f'- **({LETTERS[i]})** {value}' for i, value in enumerate(options))
        if not option_lines:
            option_lines = '_Free-response item; no options._'
        factual = row.get('factual_answer') or '_not supplied_'
        target = row.get('behavioral_target') or '_not supplied_'
        metadata = ' · '.join(
            f'{name}={row.get(name)}' for name in ['split', 'task_type', 'source', 'gen_fn', 'difficulty', 'arm']
            if row.get(name) not in (None, '', np.nan)
        )
        return (
            f"### `{row.get('id', 'no-id')}`\n\n"
            f"**Artifact:** `{row.get('artifact')}`  \n"
            f"**Metadata:** {metadata or '_none_'}  \n"
            f"**Session key:** `{row.get('key_string', '')}`\n\n"
            f"**Question:** {row.get('question', '')}\n\n"
            f"{option_lines}\n\n"
            f"**Factual answer:** {factual}  \n"
            f"**Behavioral target:** {target}"
        )

    for artifact in ARTIFACTS:
        subset = df[df['artifact'].eq(artifact)]
        display(Markdown(f'## {artifact} — {len(subset):,} records'))
        if subset.empty:
            display(Markdown('_No samples available: this artifact is empty or missing._'))
            continue
        sampled = subset.sample(min(EXAMPLES_PER_ARTIFACT, len(subset)), random_state=RANDOM_SEED)
        for row in sampled.to_dict('records'):
            display(Markdown(render_example(row)))

### 4. Schema coverage and missingness

In [ ]:
if FINAL_DATA_AVAILABLE:
    core_fields = [
        'id', 'pair_id', 'split', 'artifact', 'evaluation_group', 'task_type', 'source',
        'gen_fn', 'difficulty', 'arm', 'question', 'options', 'answer', 'correct_index',
        'target_index', 'target_letter', 'key_string', 'meta'
    ]
    coverage = []
    for artifact, group in df.groupby('artifact', sort=False):
        for field in core_fields:
            present = field in group and group[field].notna().sum()
            coverage.append({
                'artifact': artifact,
                'field': field,
                'non_null': int(present),
                'records': len(group),
                'coverage_pct': 100 * present / len(group),
            })
    coverage_df = pd.DataFrame(coverage)
    coverage_pivot = coverage_df.pivot(index='field', columns='artifact', values='coverage_pct')
    display(coverage_pivot.round(1))

    missing_summary = (
        coverage_df[coverage_df['coverage_pct'].lt(100)]
        .sort_values(['coverage_pct', 'artifact', 'field'])
    )
    display(Markdown('### Fields with incomplete coverage'))
    display(missing_summary if not missing_summary.empty else Markdown('_All audited fields have complete coverage._'))

### 5. Category and label distributions

In [ ]:
if FINAL_DATA_AVAILABLE:
    def distribution_table(column, normalize=False):
        table = pd.crosstab(
            df[column].fillna('∅ missing'),
            df['evaluation_group'],
            normalize='columns' if normalize else False,
            margins=not normalize,
        )
        return table.mul(100) if normalize else table

    categorical_columns = [
        'split', 'task_type', 'source', 'gen_fn', 'difficulty', 'option_kind',
        'arm', 'correct_letter', 'target_letter', 'option_count', 'target_matches_fact'
    ]
    for column in categorical_columns:
        if column not in df or df[column].notna().sum() == 0:
            continue
        display(Markdown(f'### {column} — counts'))
        display(distribution_table(column))
        display(Markdown(f'**{column} — within-group percentages**'))
        display(distribution_table(column, normalize=True).round(1))

In [ ]:
if FINAL_DATA_AVAILABLE:
    if HAS_PLOTS:
        plot_columns = [c for c in ['artifact', 'task_type', 'gen_fn', 'arm', 'correct_letter', 'target_letter'] if df[c].notna().any()]
        fig, axes = plt.subplots(math.ceil(len(plot_columns) / 2), 2, figsize=(16, 4.5 * math.ceil(len(plot_columns) / 2)))
        axes = np.atleast_1d(axes).ravel()
        for ax, column in zip(axes, plot_columns):
            order = df[column].fillna('∅ missing').value_counts().index
            sns.countplot(data=df.assign(**{column: df[column].fillna('∅ missing')}), y=column, order=order, ax=ax, color='#4472C4')
            ax.set_title(f'{column} distribution')
            ax.set_xlabel('records')
        for ax in axes[len(plot_columns):]:
            ax.axis('off')
        plt.tight_layout()
        plt.show()

### 6. Text lengths: characters, words, and tokens

In [ ]:
if FINAL_DATA_AVAILABLE:
    length_columns = [
        'question_chars', 'question_words', 'question_tokens',
        'factual_answer_chars', 'factual_answer_words', 'factual_answer_tokens',
        'behavioral_target_chars', 'behavioral_target_words', 'behavioral_target_tokens',
        'options_chars', 'options_words', 'options_tokens',
        'prompt_chars', 'prompt_words', 'prompt_tokens',
        'key_string_chars', 'key_string_tokens', 'max_option_chars',
    ]
    length_summary = (
        df.groupby('evaluation_group')[length_columns]
        .agg(['count', 'mean', 'median', 'std', 'min', lambda s: s.quantile(.95), 'max'])
    )
    length_summary.columns = [f'{metric}__{stat if isinstance(stat, str) else "p95"}' for metric, stat in length_summary.columns]
    display(length_summary.T.round(1))

    compact_length_summary = (
        df.groupby('evaluation_group')
        .agg(
            records=('id', 'size'),
            question_words_mean=('question_words', 'mean'),
            question_words_p95=('question_words', lambda s: s.quantile(.95)),
            prompt_tokens_mean=('prompt_tokens', 'mean'),
            prompt_tokens_p95=('prompt_tokens', lambda s: s.quantile(.95)),
            prompt_tokens_max=('prompt_tokens', 'max'),
            answer_words_mean=('factual_answer_words', 'mean'),
        )
        .sort_values('records', ascending=False)
    )
    display(Markdown('### Compact split comparison'))
    display(compact_length_summary.round(1))

In [ ]:
if FINAL_DATA_AVAILABLE:
    if HAS_PLOTS:
        metrics_to_plot = ['question_words', 'question_tokens', 'prompt_tokens', 'factual_answer_tokens', 'options_tokens', 'key_string_tokens']
        fig, axes = plt.subplots(3, 2, figsize=(17, 14))
        for ax, metric in zip(axes.ravel(), metrics_to_plot):
            sns.histplot(data=df, x=metric, hue='evaluation_group', element='step', stat='density', common_norm=False, ax=ax)
            ax.set_title(metric.replace('_', ' ').title())
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(15, 6))
        sns.boxplot(data=df, x='prompt_tokens', y='evaluation_group', showfliers=False)
        plt.title(f'Prompt token lengths by evaluation group — {token_method}')
        plt.xlabel('tokens')
        plt.ylabel('')
        plt.tight_layout()
        plt.show()

### 7. Pair integrity, duplicates, and cross-split leakage

The normalized hash intentionally excludes session keys, answer targets, and arm so that the same underlying question can be detected across artifacts.

In [ ]:
if FINAL_DATA_AVAILABLE:
    def normalize_text(value):
        return re.sub(r'\s+', ' ', str(value or '').strip().lower())

    def content_hash(row):
        payload = {
            'question': normalize_text(row.get('question')),
            'options': [normalize_text(x) for x in safe_list(row.get('options'))],
        }
        encoded = json.dumps(payload, sort_keys=True, ensure_ascii=False).encode('utf-8')
        return hashlib.sha256(encoded).hexdigest()

    df['content_hash'] = [content_hash(row) for row in df.to_dict('records')]

    id_dupes = df[df.duplicated('id', keep=False)].sort_values('id') if 'id' in df else pd.DataFrame()
    display(Markdown(f'**Duplicate IDs:** {id_dupes["id"].nunique() if not id_dupes.empty else 0:,}'))
    if not id_dupes.empty:
        display(id_dupes[['id', 'artifact', 'split', 'arm']])

    hash_artifacts = df.groupby('content_hash')['artifact'].agg(lambda x: sorted(set(x)))
    cross_artifact_hashes = hash_artifacts[hash_artifacts.map(len).gt(1)]
    display(Markdown(f'**Underlying question hashes appearing in multiple artifacts:** {len(cross_artifact_hashes):,}'))
    if len(cross_artifact_hashes):
        leak_rows = (
            df[df['content_hash'].isin(cross_artifact_hashes.index)]
            [['content_hash', 'artifact', 'split', 'id', 'pair_id', 'question']]
            .sort_values(['content_hash', 'artifact'])
        )
        display(leak_rows.head(50))

    pair_stats = (
        df.dropna(subset=['pair_id'])
        .groupby(['artifact', 'pair_id'])
        .agg(rows=('id', 'size'), arms=('arm', lambda x: tuple(sorted(set(x.dropna())))), questions=('content_hash', 'nunique'))
        .reset_index()
    )
    pair_issues = pair_stats[(pair_stats['questions'].ne(1)) | (~pair_stats['rows'].isin([1, 2]))]
    display(Markdown(f'**Pair groups with unexpected row count or multiple questions:** {len(pair_issues):,}'))
    display(pair_issues.head(50) if not pair_issues.empty else Markdown('_No structural pair issues detected._'))

    display(Markdown('### Pair-size distribution by artifact'))
    display(pd.crosstab(pair_stats['rows'], pair_stats['artifact'], margins=True))

In [ ]:
if FINAL_DATA_AVAILABLE:
    pair_consistency = []
    for (artifact, pair_id), group in df.dropna(subset=['pair_id']).groupby(['artifact', 'pair_id']):
        pair_consistency.append({
            'artifact': artifact,
            'pair_id': pair_id,
            'rows': len(group),
            'unique_questions': group['content_hash'].nunique(),
            'unique_correct_indices': group['correct_index'].nunique(dropna=True) if 'correct_index' in group else 0,
            'unique_option_lists': group['options_list'].map(json.dumps).nunique(),
            'unique_keys': group['key_string'].nunique(dropna=True),
            'arms': ', '.join(sorted(map(str, group['arm'].dropna().unique()))),
        })
    pair_consistency = pd.DataFrame(pair_consistency)
    broken_pairs = pair_consistency[
        pair_consistency['unique_questions'].gt(1)
        | pair_consistency['unique_correct_indices'].gt(1)
        | pair_consistency['unique_option_lists'].gt(1)
    ]
    display(Markdown(f'**Pairs disagreeing on question/options/factual answer:** {len(broken_pairs):,}'))
    display(broken_pairs.head(50) if not broken_pairs.empty else Markdown('_All paired rows agree on their shared factual content._'))

### 8. Outliers and records for manual review

In [ ]:
if FINAL_DATA_AVAILABLE:
    review_columns = [
        'artifact', 'id', 'task_type', 'gen_fn', 'arm', 'question',
        'factual_answer', 'behavioral_target', 'question_words', 'prompt_tokens'
    ]

    for metric in ['question_words', 'prompt_tokens', 'factual_answer_words', 'max_option_chars']:
        display(Markdown(f'### Largest `{metric}`'))
        display(df.nlargest(min(10, len(df)), metric)[review_columns + ([metric] if metric not in review_columns else [])])

    display(Markdown('### Very short or blank questions'))
    short_questions = df[df['question_words'].le(3)].sort_values('question_words')
    display(short_questions[review_columns].head(50) if not short_questions.empty else Markdown('_None found._'))

    display(Markdown('### MCQ records with unexpected option counts'))
    unexpected_options = df[df['options'].notna() & df['option_count'].ne(4)] if 'options' in df else pd.DataFrame()
    display(unexpected_options[review_columns + ['option_count']].head(50) if not unexpected_options.empty else Markdown('_None found._'))

### 9. Automated audit summary

In [ ]:
if FINAL_DATA_AVAILABLE:
    summary_lines = [
        '# EDA summary',
        f'- Loaded **{len(df):,} records** from **{df["artifact"].nunique():,} non-empty artifacts**.',
        f'- Token counts use **{token_method}**.',
        f'- **{len(empty_expected)}** expected artifacts are present but empty; **{len(missing_expected)}** are missing.',
        f'- Found **{id_dupes["id"].nunique() if not id_dupes.empty else 0:,} duplicate IDs**.',
        f'- Found **{len(cross_artifact_hashes):,} underlying questions shared across artifacts**.',
        f'- Found **{len(broken_pairs):,} paired examples that disagree on question, options, or factual answer**.',
        f'- Median prompt length is **{df["prompt_tokens"].median():,.0f} tokens**; p95 is **{df["prompt_tokens"].quantile(.95):,.0f}**; maximum is **{df["prompt_tokens"].max():,.0f}**.',
    ]

    if df['target_matches_fact'].notna().any():
        mismatch_rate = 100 * (1 - df['target_matches_fact'].dropna().mean())
        summary_lines.append(f'- The behavioral target differs from the factual answer in **{mismatch_rate:.1f}%** of comparable MCQ records.')
    if empty_expected:
        summary_lines.append('- Empty artifacts: ' + ', '.join(f'`{x}`' for x in empty_expected) + '.')
    if cross_artifact_hashes.empty:
        summary_lines.append('- No exact normalized question/option leakage was detected across artifacts.')

    display(Markdown('\n'.join(summary_lines)))

### Optional next steps

- Set `TOKENIZER_NAME` to the exact training tokenizer and rerun all cells before using token limits operationally.
- Re-run this notebook after the currently empty grounded and held-out artifacts are supplied.
- Add semantic near-duplicate detection (for example, embeddings or MinHash) if exact normalized hashes are insufficient.
- Export the `review_columns` tables to CSV only when a persistent audit artifact is needed.